In [37]:
import pandas as pd
data = pd.read_csv('../data/nova_pay_combined.csv')

In [38]:
data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
data['amount_src'] = data['amount_src'].str.replace(',', '').astype(float)

In [39]:
data.info()
raw = pd.read_csv('../data/nova_pay_combined.csv')['timestamp']
raw[data['timestamp'].isna() & raw.notna()].unique()[:20]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  object             
 1   customer_id                11400 non-null  object             
 2   timestamp                  11339 non-null  datetime64[ns, UTC]
 3   home_country               11400 non-null  object             
 4   source_currency            11400 non-null  object             
 5   dest_currency              11400 non-null  object             
 6   channel                    11400 non-null  object             
 7   amount_src                 11400 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  de

array(['2025/13/40 25:61:00', '0000-00-00T00:00:00Z'], dtype=object)

In [40]:
print("Duplicate rows:", data.duplicated().sum())
print("Duplicate transaction IDs:", data['transaction_id'].duplicated().sum())

Duplicate rows: 200
Duplicate transaction IDs: 200


In [41]:
data = data.drop_duplicates()
data.shape

(11200, 26)

In [42]:
print("Duplicate rows:", data.duplicated().sum())

Duplicate rows: 0


In [43]:
for col in ['home_country', 'source_currency', 'dest_currency', 'channel', 'ip_country', 'kyc_tier']:
    print(data[col].value_counts(dropna=False), '\n')

home_country
US         7784
UK         2091
CA         1205
 US          68
unknown      32
 UK          13
 CA           7
Name: count, dtype: int64 

source_currency
USD    7875
GBP    2111
CAD    1214
Name: count, dtype: int64 

dest_currency
NGN    1447
INR    1304
USD    1302
CNY    1297
PHP    1290
GBP    1273
CAD    1260
EUR    1177
MXN     850
Name: count, dtype: int64 

channel
mobile       6242
web          3674
ATM           989
mobille        59
 mobile        48
MOBILE         46
unknown        37
WEB            35
 web           33
weeb           24
ATm             9
 ATM            4
Name: count, dtype: int64 

ip_country
US         6713
UK         2372
CA         1703
NaN         296
 US          48
unknown      32
 UK          18
 CA          15
NAN           2
 nan          1
Name: count, dtype: int64 

kyc_tier
standard       7780
enhanced       1794
low            1034
NaN             295
standrd          72
STANDARD         69
 standard        64
unknown          

In [44]:
import numpy as np

text_cols = ['home_country', 'source_currency', 'dest_currency', 'channel', 'ip_country', 'kyc_tier']

for col in text_cols:
    data[col] = data[col].str.strip()
    data.loc[data[col].str.lower().isin(['unknown', 'nan']), col] = np.nan

for col in ['home_country', 'source_currency', 'dest_currency', 'ip_country']:
    data[col] = data[col].str.upper()

data['channel'] = data['channel'].str.lower()
data['kyc_tier'] = data['kyc_tier'].str.lower().replace({'standrd': 'standard', 'enhancd': 'enhanced'})

In [45]:
for col in ['home_country', 'source_currency', 'dest_currency', 'channel', 'ip_country', 'kyc_tier']:
    print(data[col].value_counts(dropna=False), '\n')

home_country
US     7852
UK     2104
CA     1212
NaN      32
Name: count, dtype: int64 

source_currency
USD    7875
GBP    2111
CAD    1214
Name: count, dtype: int64 

dest_currency
NGN    1447
INR    1304
USD    1302
CNY    1297
PHP    1290
GBP    1273
CAD    1260
EUR    1177
MXN     850
Name: count, dtype: int64 

channel
mobile     6336
web        3742
atm        1002
mobille      59
NaN          37
weeb         24
Name: count, dtype: int64 

ip_country
US     6761
UK     2390
CA     1718
NaN     331
Name: count, dtype: int64 

kyc_tier
standard    7985
enhanced    1839
low         1047
NaN          329
Name: count, dtype: int64 



In [46]:
data['channel'] = data['channel'].replace({'mobille': 'mobile', 'weeb': 'web'})
data['channel'].value_counts(dropna=False)

channel
mobile    6395
web       3766
atm       1002
NaN         37
Name: count, dtype: int64

In [47]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
amount_src,11200.0,437.572747,1373.086218,-9997.160,90.7150,158.980000,295.042500,11942.890000
amount_usd,10900.0,451.710904,1401.028359,7.230,92.4925,163.485000,302.707500,12498.570000
fee,10910.0,99.212001,952.526927,-1.000,2.3800,3.500000,5.560000,9999.990000
exchange_rate_src_to_dest,11200.0,167.453693,381.930143,0.592,1.0000,7.142857,73.529412,1388.888889
ip_risk_score,11200.0,0.397541,0.271213,0.004,0.2090,0.325000,0.488000,1.200000
account_age_days,11200.0,392.848571,342.123998,1.000,147.0000,293.000000,661.000000,1095.000000
device_trust_score,10910.0,0.653238,0.273174,-0.100,0.5150,0.654500,0.894000,0.999000
chargeback_history_count,11200.0,0.049375,0.258388,0.000,0.0000,0.000000,0.000000,2.000000
risk_score_internal,11200.0,0.267653,0.143521,0.000,0.1690,0.223000,0.391000,0.900000
txn_velocity_1h,11200.0,0.463839,1.532671,-1.000,0.0000,0.000000,0.000000,8.000000


In [49]:
print("Negative amount_src:", (data['amount_src'] < 0).sum())
print("Bad fee:", ((data['fee'] < 0) | (data['fee'] == 9999.99)).sum())
print("ip_risk_score > 1:", (data['ip_risk_score'] > 1).sum())
print("device_trust_score < 0:", (data['device_trust_score'] < 0).sum())
print("txn_velocity_1h < 0:", (data['txn_velocity_1h'] < 0).sum())

Negative amount_src: 0
Bad fee: 0
ip_risk_score > 1: 0
device_trust_score < 0: 0
txn_velocity_1h < 0: 0


In [50]:
data.loc[data['amount_src'] < 0, 'amount_src'] = np.nan
data.loc[(data['fee'] < 0) | (data['fee'] == 9999.99), 'fee'] = np.nan
data.loc[data['ip_risk_score'] > 1, 'ip_risk_score'] = np.nan
data.loc[data['device_trust_score'] < 0, 'device_trust_score'] = np.nan
data.loc[data['txn_velocity_1h'] < 0, 'txn_velocity_1h'] = np.nan

In [51]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
amount_src,11100.0,444.331561,1374.173818,7.230,92.2025,160.490000,297.890000,11942.890000
amount_usd,10900.0,451.710904,1401.028359,7.230,92.4925,163.485000,302.707500,12498.570000
fee,10710.0,7.703448,20.702317,0.500,2.4000,3.500000,5.500000,179.740000
exchange_rate_src_to_dest,11200.0,167.453693,381.930143,0.592,1.0000,7.142857,73.529412,1388.888889
ip_risk_score,11000.0,0.382951,0.250942,0.004,0.2070,0.321000,0.476000,1.000000
account_age_days,11200.0,392.848571,342.123998,1.000,147.0000,293.000000,661.000000,1095.000000
device_trust_score,10710.0,0.667305,0.255389,0.005,0.5220,0.702000,0.894000,0.999000
chargeback_history_count,11200.0,0.049375,0.258388,0.000,0.0000,0.000000,0.000000,2.000000
risk_score_internal,11200.0,0.267653,0.143521,0.000,0.1690,0.223000,0.391000,0.900000
txn_velocity_1h,11000.0,0.490455,1.533663,0.000,0.0000,0.000000,0.000000,8.000000


In [2]:
import pandas as pd
raw_df = pd.read_csv('../data/nova_pay_combined.csv')
high_fee = raw_df[raw_df['fee'] >= 1000]
print("Count:", len(high_fee))
print(high_fee['fee'].value_counts())

Count: 103
fee
9999.99    103
Name: count, dtype: int64


In [3]:
for col in ['channel', 'source_currency', 'dest_currency', 'kyc_tier', 'is_fraud']:
    print(high_fee[col].value_counts(), '\n')

channel
mobile       57
web          33
ATM          10
WEB           1
 mobile       1
mobille       1
Name: count, dtype: int64 

source_currency
USD    66
GBP    22
CAD    15
Name: count, dtype: int64 

dest_currency
PHP    20
USD    18
CAD    15
GBP    12
EUR    10
NGN     9
CNY     9
INR     6
MXN     4
Name: count, dtype: int64 

kyc_tier
standard    79
enhanced    10
low          8
standrd      2
ENHANCED     1
Name: count, dtype: int64 

is_fraud
0    99
1     4
Name: count, dtype: int64 

